In [ ]:
!git clone https://github.com/seznam/MLPrague-2026.git
!pip install ./MLPrague-2026
!pip install torch-geometric
!cp -a MLPrague-2026/images .
!cp -a MLPrague-2026/data .

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import Markdown, display, HTML
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from ml_prague_2026.evaluation import compare_models, evaluate_model
from ml_prague_2026.losses import FocalLoss, calculate_class_weights
from ml_prague_2026.utils import create_undirected_edge_index, stratified_split_indices, indices_to_mask
from ml_prague_2026.models import train_chart

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import ModuleList
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.seed import seed_everything

seed_everything(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

assert device == 'cuda', "can't access cuda (GPU) device"

# Supervised Anomaly Detection on Graphs — Hands-on Workshop

In this notebook you will progressively integrate graph structure into **supervised** anomaly detection task on the **YelpChi** fraud-review dataset.

**Approaches we will train & compare:**
- **Manual graph features + Random Forest classifier** — compute topological features, combine with tabular features
- **Node2Vec embeddings + Random Forest classifier** — learn node representations from graph structure
- **Graph Convolutional Network** — end-to-end learning on the homogeneous graph
- **Heterogeneous Graph Convolutional Network** — separate convolutions per edge type

## 1. Data loading & graph construction

The **YelpChi** dataset contains 46k Yelp reviews of Chicago businesses (like restaurants etc.) with hand-crafted features and a binary `spam` label (see the introduction notebook for full exploration).

Reviews are connected through three types of relations:
- Review — User — Review - **R-U-R** - two reviews by the **same user** (~49k edges)
- Review — Star — Review - **R-S-R** - two reviews of the **same business** with the **same star rating** (~3.4M edges)
- Review — Time — Review - **R-T-R** - two reviews of the **same business** in the **same month** (~574k edges)

&nbsp;

<center><img src="images/relations.png"></center>

In [ ]:
# Dataset paths
BASE_PATH = ''

YELP_CHI_PATH = BASE_PATH + 'data/yelpchi.parquet'

YELP_CHI_EDGE_RUR_PATH = BASE_PATH + 'data/yelpchi_rur.npy'
YELP_CHI_EDGE_RSR_PATH = BASE_PATH + 'data/yelpchi_rsr.npy'
YELP_CHI_EDGE_RTR_PATH = BASE_PATH + 'data/yelpchi_rtr.npy'

# Load the dataset
yelp_chi = pd.read_parquet(YELP_CHI_PATH)
FEATURES = [c for c in yelp_chi.columns if c.startswith('f_')]

display(yelp_chi.head(3))
print()
print(f'Reviews: {len(yelp_chi):,}, Features: {len(FEATURES)}, Spam ratio: {yelp_chi["spam"].mean():.1%}')

### TASK 1.A: Build the R-U-R (Review-User-Review) edge list

If multiple reviews were written by the same user, that's a useful signal — a fraudulent user likely writes multiple fake reviews. We capture this by **connecting reviews that share an author** with an R-U-R (Review-User-Review) edge.

Build an edge list — a numpy array of shape `(num_edges, 2)` — containing pairs of `review_id`s that share the same `user_id`.

In [ ]:
# TASK 1.A: build R-U-R edges from user_id
# - each row in yelp_chi has a 'review_id' and 'user_id'
# - connect all pairs of reviews that share the same user_id
# - result should be a numpy array of shape (num_edges, 2)
#
# - Hint: Use a self-merge of reviews on `user_id`, then filter out self-loops (a review paired with itself)
#         and duplicate edges (those in the opposite direction, if any), e.g. using comparison on their review_id columns.

edges_rur = None  ### YOUR CODE HERE ###

<!-- #region -->
<details>
<summary><b>Task 1.A — Solution</b></summary>

```python
edges_rur = (
    yelp_chi[['review_id', 'user_id']]
        .merge(yelp_chi[['review_id', 'user_id']], on='user_id')
        .query('review_id_x < review_id_y')
        [['review_id_x', 'review_id_y']]
        .values
)
```

</details>
<!-- #endregion -->

In [ ]:
print(f'R-U-R edges built: {len(edges_rur):,}')

# Load pre-computed edge lists for the other two relation types
edges_rsr = np.load(YELP_CHI_EDGE_RSR_PATH, allow_pickle=True)
edges_rtr = np.load(YELP_CHI_EDGE_RTR_PATH, allow_pickle=True)
print(f'R-S-R edges loaded: {len(edges_rsr):,}')
print(f'R-T-R edges loaded: {len(edges_rtr):,}')

In [ ]:
num_nodes = len(yelp_chi)

X_tab = yelp_chi[FEATURES].values
y_all = yelp_chi['spam'].values

# Stratified train/val/test split over node indices (used by all classifiers - RF, GNNs, etc.)
idx_train, idx_val, idx_test = stratified_split_indices(y_all, train_ratio=0.7, val_ratio=0.1)
print(f'Train: {len(idx_train):,}, Val: {len(idx_val):,}, Test: {len(idx_test):,}')

np.savez(BASE_PATH + 'data/yelpchi_split.npz', train=idx_train, val=idx_val, test=idx_test)

X_train, X_val, X_test = X_tab[idx_train], X_tab[idx_val], X_tab[idx_test]
y_train, y_val, y_test = y_all[idx_train], y_all[idx_val], y_all[idx_test]

### Random Forest Baseline
Train Random Forest on tabular features only - no graph information

In [ ]:
def random_forest(X_train, y_train, X_test, random_state=42):
    """Train a RandomForest on (X_train, y_train) and predict X_test."""
    rf = RandomForestClassifier(random_state=random_state)
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)
    preds = rf.classes_[proba.argmax(axis=1)]
    return preds, proba[:, 1]

In [ ]:
# Train Random Forest on tabular features only
_preds, _probs = random_forest(X_train, y_train, X_test)
baseline_results = evaluate_model('RF (tabular)', y_test, _preds, _probs)

## 2. Manual graph features + ML classifier

The simplest way to leverage graph structure is to compute **topological features** for each node and add them as extra columns alongside the existing 30 features.

The most basic graph feature is **degree** — the number of edges connected to a node. Since we have three edge types, we can compute a separate degree for each, capturing how connected a review is through user, rating, and temporal relations.

<center><img src="images/degree.png" width="400"></center>

### TASK 2.A: Compute degree features

The **degree** of a node is the number of edges connected to it. Since we have three edge types, compute a separate degree for each.

In [ ]:
# TASK 2.A: Complete function computing node degree and apply it to each edge type
# - edges_rur, edges_rsr, edges_rtr each have shape (num_edges, 2) with numerical review_id
# - compute degree_rur, degree_rsr, degree_rtr as arrays of length num_nodes
#
# - Hint: Count how often each node index appears in the edge array. Each edge contributes to the degree of both its endpoints.
#         There are many possible ways — one is using value_counts() from pandas and then reindex to range of numerical indices.

def compute_degree(edges, num_nodes):
    ### YOUR CODE HERE ###
    return None  ### MODIFY THIS ###

degree_rur = compute_degree(edges_rur, num_nodes)
degree_rsr = compute_degree(edges_rsr, num_nodes)
degree_rtr = compute_degree(edges_rtr, num_nodes)

<!-- #region -->
<details>
<summary><b>Task 2.A — Solution</b></summary>

There are many possible solutions. Here is one using pandas:

```python
def compute_degree(edges, num_nodes):
    node_ids = [node for edge in edges for node in edge]
    return (pd.Series(node_ids)
            .value_counts()
            .reindex(range(num_nodes), fill_value=0)
            .values)
```

</details>
<!-- #endregion -->

In [ ]:
# Degree distributions by class (clipped at 99th percentile for readability)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, deg, name in zip(axes, [degree_rur, degree_rsr, degree_rtr], ['R-U-R', 'R-S-R', 'R-T-R']):
    q99 = np.quantile(deg, 0.99)
    bins = np.linspace(0, q99, 51)
    ax.hist(deg[y_all == 0].clip(max=q99), bins=bins, density=True, color='steelblue', label='normal')
    ax.hist(deg[y_all == 1].clip(max=q99), bins=bins, density=True, color='salmon', label='spam', alpha=0.7)
    ax.set_title(f'{name} degree')
    ax.legend()
plt.tight_layout()
plt.show();

### Random Forest with Degree Features
Train Random Forest with degree features only - no tabular data.

In [ ]:
# Use only degree features
X_degree = np.column_stack([degree_rur, degree_rsr, degree_rtr])

# RF with degree features only
_preds, _probs = random_forest(X_degree[idx_train], y_train, X_degree[idx_test])
degree_results = evaluate_model('RF (degree)', y_test, _preds, _probs)

### Random Forest with Degree Features and Tabular Data
Combine degree with tabular features.

In [ ]:
# Combine tabular features with degree features
X_train_deg_comb = np.concatenate([X_train, X_degree[idx_train]], axis=1)
X_test_deg_comb = np.concatenate([X_test, X_degree[idx_test]], axis=1)

# RF with tabular + degree features
_preds, _probs = random_forest(X_train_deg_comb, y_train, X_test_deg_comb)
degree_results_comb = evaluate_model('RF (tabular + degree)', y_test, _preds, _probs)

In [ ]:
compare_models([baseline_results, degree_results, degree_results_comb]);

## 3. Node2Vec + ML classifier

Instead of hand-crafting graph features, we can **learn** node representations automatically.

[Node2Vec](https://arxiv.org/abs/1607.00653) performs biased random walks on the graph and uses a Skip-gram model (similar to Word2Vec) to learn embeddings that capture both local and global graph structure.

**How Node2Vec works:**
1. Perform random walks starting from each node (controlled by parameters `p` and `q`)
2. Treat walks as "sentences" and nodes as "words"
3. Apply Word2Vec (Skip-gram) to learn node embeddings

<center><img src="images/node2vec.png" width="600"></center>

&nbsp;

Parameters `p` (return) and `q` (in-out) control the walk behavior:
- `q` → local vs global exploration
- `p` → BFS vs DFS-like walks

In [ ]:
# Kept for reference only, we will actually use precomputed embeddings due to time constraints, see the next cell.

# karateclub installation for Node2Vec.
#
#! pip install karateclub==1.2.0 --force-reinstall --no-cache-dir

# Build NetworkX graph for Node2Vec (requires undirected NetworkX graph)
#
# import networkx as nx
# from karateclub import Node2Vec
#
# G = nx.Graph()
# G.add_nodes_from(range(num_nodes))
# G.add_edges_from(zip(src, dst))
# print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

# Fit Node2Vec: random walks + Skip-gram to learn 32-dim node embeddings
#
# node2vec = Node2Vec(walk_length=40, walk_number=40, dimensions=32, p=0.25, q=0.5, workers=8, seed=42)
# node2vec.fit(G)
# node2vec_embeddings = node2vec.get_embedding()
# np.save(BASE_PATH + 'data/node2vec_embeddings.npy', node2vec_embeddings)

In [ ]:
node2vec_embeddings = np.load(BASE_PATH + 'data/node2vec_embeddings.npy')

### Random Forest with Node2Vec Embeddings

In [ ]:
# Train RF on Node2Vec embeddings only (no tabular features)
X_emb_train = node2vec_embeddings[idx_train]
X_emb_test = node2vec_embeddings[idx_test]

# RF with embeddings
_preds, _probs = random_forest(X_emb_train, y_train, X_emb_test)
emb_results = evaluate_model('RF (embedding)', y_test, _preds, _probs)

### Random Forest with Node2Vec Embeddings and Tabular Features

In [ ]:
# Node2Vec embeddings + tabular features combined
X_comb_train = np.concatenate([X_train, X_emb_train], axis=1)
X_comb_test = np.concatenate([X_test, X_emb_test], axis=1)

# RF with embeddings
_preds, _probs = random_forest(X_comb_train, y_train, X_comb_test)
combined_results = evaluate_model('RF (tabular + embedding)', y_test, _preds, _probs)

In [ ]:
compare_models([baseline_results, degree_results, degree_results_comb, emb_results, combined_results]);

## 4. Graph Convolutional Network

One of the simplest GNN architectures is a simple **Graph Convolutional Network** where each layer aggregates features from all neighbors with learnable weights, then applies a nonlinearity.

After stacking multiple layers, each node's representation encodes information from its multi-hop neighborhood. The model learns both the features and the classifier **end-to-end** — directly optimizing for the classification task. We'll use [GraphSAGE](https://arxiv.org/abs/1706.02216) convolutions here.

### Data preparation

In [ ]:
# Combine all three edge types into one undirected edge_index for the homogeneous graph
edges_all = np.concatenate([edges_rur, edges_rsr, edges_rtr], axis=0)
edge_index = create_undirected_edge_index(edges_all)

# Node features and labels as torch tensors
x = torch.tensor(X_tab, dtype=torch.float)
y = torch.tensor(y_all, dtype=torch.long)

# Create the PyG Data object — the standard container for a homogeneous graph in PyTorch Geometric
data = Data(x=x, edge_index=edge_index, y=y)

# Apply the stratified split from section 1 as boolean train/val/test masks over nodes
data.train_mask = indices_to_mask(idx_train, data.num_nodes)
data.val_mask = indices_to_mask(idx_val, data.num_nodes)
data.test_mask = indices_to_mask(idx_test, data.num_nodes)

data = data.to(device)
data

### TASK 4.A: Define the convolutional layers

<center><img src="images/gcn-vis.png"></center>

Complete the `__init__` method of the `SAGE` class by creating `self.convs` — a [ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html) of `num_layers` [SAGEConv](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.SAGEConv.html) layers.

The output projection head and the `forward` method are already provided.

In [ ]:
class SAGE(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=96, out_dim=2, num_layers=2, dropout=0.24):
        super().__init__()
        self.dropout_rate = dropout

        # TASK 4.A: Define the convolutional layers
        # - create self.convs as a ModuleList of num_layers SAGEConv layers
        # - the first layer maps in_dim → hidden_dim
        # - the remaining layers map hidden_dim → hidden_dim
        # - (where the in_dim is size of our input samples, i.e. the number of features)
        # - Note: torch.nn.ModuleList and torch_geometric.nn.SAGEConv have been imported for you
        #
        # - Hint: Create an empty ModuleList, append the first layer, then append the rest in a loop.
        #
        #
        self.convs = None  ### MODIFY THIS ###
        #
        ### YOUR CODE HERE ###

        self.classifier = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.mish(x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)
        return self.classifier(x)

<!-- #region -->
<details>
<summary><b>Task 4.A — Solution</b></summary>

```python
self.convs = ModuleList()
self.convs.append(SAGEConv(in_dim, hidden_dim))
for _ in range(num_layers - 1):
    self.convs.append(SAGEConv(hidden_dim, hidden_dim))
```

</details>
<!-- #endregion -->

In [ ]:
def train_gnn(model, data, criterion, epochs=500, lr=0.003, log_interval=50):
    """Train a GNN model with given loss function (the `criterion` parameter). Returns training history."""
    seed_everything(42)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            train_loss = criterion(out[data.train_mask], data.y[data.train_mask])
            val_loss = criterion(out[data.val_mask], data.y[data.val_mask])
            pred = out.argmax(dim=1)
            train_f1 = f1_score(data.y[data.train_mask].cpu(), pred[data.train_mask].cpu(), average='macro')
            val_f1 = f1_score(data.y[data.val_mask].cpu(), pred[data.val_mask].cpu(), average='macro')

        history['train_loss'].append(train_loss.item())
        history['val_loss'].append(val_loss.item())
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        if epoch == 1 or epoch % log_interval == 0:
            print(f'Epoch {epoch:03d} | Train loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}')

    return history


def evaluate_gnn(name, model, data):
    """Evaluate a trained GNN on the test mask."""
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(data.x, data.edge_index), dim=1)[:, 1]
    mask = data.test_mask
    return evaluate_model(name, data.y[mask].cpu().numpy(), (probs[mask] > 0.5).int().cpu().numpy(), probs[mask].cpu().numpy())

### Handling class imbalance

Class imbalance is common in anomaly detection tasks, and it can be a challenge for training GNNs. Without any adjustments, the model may learn to simply predict the majority class, although in our case it is not so bad (14.5% spam vs 85.5% normal).

Approaches we will use:
- **Class weights** — weighting the minority _class_ higher
- **Focal Loss** — down-weighting easy _examples_, focusing on hard ones

We'll use class weights here and focal loss in the next section.

In [ ]:
class_weights = calculate_class_weights(data)
print(f'Class weights: {class_weights}')

In [ ]:
sage = SAGE(data.num_features).to(device)
history_sage = train_gnn(sage, data, criterion=torch.nn.CrossEntropyLoss(weight=class_weights))
train_chart(history_sage)

In [ ]:
display(Markdown('**SAGE**'))
sage_results = evaluate_gnn('SAGE', sage, data);

In [ ]:
compare_models([baseline_results, degree_results, degree_results_comb, emb_results, combined_results, sage_results]);

## 5. Heterogeneous Graph Convolutional Network

In previous section we treated all edges as if they came from a single relation type, even though YelpChi actually has three distinct ones (R-U-R, R-S-R, R-T-R).

**Heterogeneous Convolutional Network** instead uses a separate `SAGEConv` per edge type and combines the per-relation outputs at each layer. This way the model can learn different aggregation weights for each relation, which is useful when different edge types carry different signals about fraud (e.g. shared user vs. shared timestamp).

We use PyG's [`HeteroData`](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.HeteroData.html) to hold the per-relation edge indices and [`HeteroConv`](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.HeteroConv.html) to wrap one `SAGEConv` per relation.

<center><img src="images/hetero-gcn-vis.png"></center>

### Heterogeneous graph construction

In [ ]:
from torch_geometric.data import HeteroData

# Build a separate undirected edge_index for each relation type
edge_index_rur = create_undirected_edge_index(edges_rur)
edge_index_rsr = create_undirected_edge_index(edges_rsr)
edge_index_rtr = create_undirected_edge_index(edges_rtr)

# Create HeteroData object — PyG's container for heterogeneous graphs (multiple node/edge types)
hetero_data = HeteroData()

# Add node features and labels for the single 'review' node type
hetero_data['review'].x = x
hetero_data['review'].y = y

# Add edge indices for each relation type
# The format is ('source_node_type', 'edge_type', 'destination_node_type')
hetero_data['review', 'rur', 'review'].edge_index = edge_index_rur
hetero_data['review', 'rsr', 'review'].edge_index = edge_index_rsr
hetero_data['review', 'rtr', 'review'].edge_index = edge_index_rtr

# Apply the same stratified split from section 1 as masks on the 'review' node type
n_review = hetero_data['review'].num_nodes
hetero_data['review'].train_mask = indices_to_mask(idx_train, n_review)
hetero_data['review'].val_mask = indices_to_mask(idx_val, n_review)
hetero_data['review'].test_mask = indices_to_mask(idx_test, n_review)

# Move the HeteroData object to the specified device (GPU)
hetero_data = hetero_data.to(device)
hetero_data

### TASK 5.A: Add a skip connection

Stacking several `HeteroConv` layers can wash out the original review features (a phenomenon known as **over-smoothing**). A common remedy is a **skip connection** — concatenating the raw input features with the final hidden representation before the classifier, so the model always has direct access to the original signal.

In `forward` method, make a snapshot of `x_dict['review']` before the conv loop and concatenate it with the final hidden state along the feature dimension.

Note: the `classifier` in `__init__` has already beed widened to `hidden_dim + in_dim` to accept the concatenated tensor.

In [ ]:
class HeteroSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim=96, out_dim=2, num_layers=4, dropout=0.23):
        super().__init__()
        self.dropout_rate = dropout

        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers):
            # use heterogeneous convolution - a separate SAGEConv for each relation type
            self.convs.append(HeteroConv({
                ('review', 'rur', 'review'): SAGEConv((-1, -1), hidden_dim, aggr='mean'),
                ('review', 'rtr', 'review'): SAGEConv((-1, -1), hidden_dim, aggr='mean'),
                ('review', 'rsr', 'review'): SAGEConv((-1, -1), hidden_dim, aggr='mean'),
            }, aggr='sum'))

        # widened to accept concatenated [hidden, input] from the skip connection (see TASK 5.A)
        self.classifier = nn.Linear(hidden_dim + in_dim, out_dim)

    def forward(self, x_dict, edge_index_dict):
        # x_dict: {'review': node_features}
        # edge_index_dict: {('review', 'rur', 'review'): edges, ...}

        # TASK 5.A: capture the raw input features and concatenate them
        #   with the final hidden state to create a skip conenction:
        # - snapshot x_dict['review'] before the conv loop into `skip_conn`
        # - after the conv loop, concatenate the snapshot with the resulting x_dict['review']
        #
        # - Hint: torch.cat([..., ...], dim=1) can be used for the concatenation along the feature dimension

        #
        ### YOUR CODE HERE ###
        #
        
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict['review'] = F.relu(x_dict['review'])
            x_dict['review'] = F.dropout(x_dict['review'], p=self.dropout_rate, training=self.training)

        x = x_dict['review']  ### MODIFY THIS ###

        return self.classifier(x)

<!-- #region -->
<details>
<summary><b>Task 5.A — Solution</b></summary>

```python
skip_conn = x_dict['review']
for conv in self.convs:
    x_dict = conv(x_dict, edge_index_dict)
    x_dict['review'] = F.relu(x_dict['review'])
    x_dict['review'] = F.dropout(x_dict['review'], p=self.dropout_rate, training=self.training)
x = torch.cat([x_dict['review'], skip_conn], dim=1)
return self.classifier(x)
```

</details>
<!-- #endregion -->

In [ ]:
def train_gnn_hetero(model, data, criterion, epochs=500, lr=0.003, log_interval=50):
    """Train a GNN model with given loss function (the `criterion` parameter). Returns training history."""
    seed_everything(42)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        # Pass x_dict and edge_index_dict for HeteroData
        out = model(data.x_dict, data.edge_index_dict)
        # Access masks and labels from the 'review' node type
        loss = criterion(out[data['review'].train_mask], data['review'].y[data['review'].train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)
            train_loss = criterion(out[data['review'].train_mask], data['review'].y[data['review'].train_mask])
            val_loss = criterion(out[data['review'].val_mask], data['review'].y[data['review'].val_mask])
            pred = out.argmax(dim=1)
            train_f1 = f1_score(data['review'].y[data['review'].train_mask].cpu(), pred[data['review'].train_mask].cpu(), average='macro')
            val_f1 = f1_score(data['review'].y[data['review'].val_mask].cpu(), pred[data['review'].val_mask].cpu(), average='macro')

        history['train_loss'].append(train_loss.item())
        history['val_loss'].append(val_loss.item())
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        if epoch == 1 or epoch % log_interval == 0:
            print(f'Epoch {epoch:03d} | Train loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}')

    return history


def evaluate_gnn_hetero(name, model, data):
    """Evaluate a trained GNN on the test mask."""
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(data.x_dict, data.edge_index_dict), dim=1)[:, 1]
    mask = data['review'].test_mask
    return evaluate_model(name, data['review'].y[mask].cpu().numpy(), (probs[mask] > 0.5).int().cpu().numpy(), probs[mask].cpu().numpy())

In [ ]:
hsage = HeteroSAGE(in_dim=hetero_data['review'].num_features).to(device)
history_hsage = train_gnn_hetero(hsage, hetero_data, FocalLoss(alpha=class_weights, gamma=1.4))
train_chart(history_hsage)

In [ ]:
display(Markdown('**Heterogeneous SAGE**'))
hsage_results = evaluate_gnn_hetero('Heterogeneous SAGE', hsage, hetero_data);

## 6. Final comparison

Let's compare all approaches side-by-side.

In [ ]:
compare_models([
    baseline_results,
    degree_results,
    degree_results_comb,
    emb_results,
    combined_results,
    sage_results,
    hsage_results,
])

## Classification examples

Let's now look at some examples of reviews classified as fraud/not-fraud by our best classifier.

In [ ]:
hsage.eval()
with torch.no_grad():
    hsage_preds = torch.softmax(hsage(hetero_data.x_dict, hetero_data.edge_index_dict), dim=1)[:, 1].cpu().numpy()

### Normal reviews

In [ ]:
EXAMPLES_NORMAL = [7696, 37814, 12186]
test_examples = yelp_chi.loc[EXAMPLES_NORMAL, ["review", "spam"]]
test_examples["prediction"] = (hsage_preds[EXAMPLES_NORMAL] > 0.5).astype(int)

for i, r in test_examples.iterrows():
    display(HTML(f"<h4>Classified non-fraud Example #{i}</h4>"))
    display(HTML(f"<b>Classified correctly?</b> {'Yes ✅' if r['spam'] == r['prediction'] else 'No ❌'}"))
    display(HTML(f"<blockquote>{r['review']}</blockquote>"))

### Fraudulent reviews

In [ ]:
EXAMPLES_FRAUD = [44708, 43100, 42660]
test_examples = yelp_chi.loc[EXAMPLES_FRAUD, ["review", "spam"]]
test_examples["prediction"] = (hsage_preds[EXAMPLES_FRAUD] > 0.5).astype(int)

for i, r in test_examples.iterrows():
    display(HTML(f"<h4>Classified Fraud Example #{i}</h4>"))
    display(HTML(f"<b>Classified correctly?</b> {'Yes ✅' if r['spam'] == r['prediction'] else 'No ❌'}"))
    display(HTML(f"<blockquote>{r['review']}</blockquote>"))

### More examples of reviews classified as fraudulent

In [ ]:
test_examples = yelp_chi[(hsage_preds > 0.5) & data.test_mask.cpu().numpy()][["review","spam"]]
for i, r in test_examples.sample(5).iterrows():
    display(HTML(f"<h4>Classified Fraud Example #{i}</h4>"))
    display(HTML(f"<b>Classified correctly?</b> {'Yes ✅' if r['spam'] else 'No ❌'}"))
    display(HTML(f"<blockquote>{r['review']}</blockquote>"))

## Summary

YelpChi dataset has strong hand-crafted features, so the tabular baselines are already very competitive. In real-world scenarios with fewer or weaker features, graph-based approaches might provide larger improvements.

| Approach | Observation |
|----------|-------------|
| **Tabular RF** | <ul><li>Establishes a **competitive baseline**. <li>Graph-based methods must contribute information beyond what the tabular features already capture. |
| **Degree features + RF** | <ul><li>Limited on their own, but combined with the tabular features they produce **the largest gain among the RF variants**. <li>Inexpensive and interpretable. |
| **Node2Vec + RF** | <ul><li>Yields **little additional benefit on this dataset**, even when combined with tabular features. <li>The embeddings are learned independently of the classification objective. |
| **Homogeneous SAGE** | <ul><li>**Does not outperform the tabular+degree RF.** <li>Merging the three edge types into a single graph dilutes the relational signal. |
| **Heterogeneous SAGE** | <ul><li>Modelling each relation separately gives the **best results**. |